# 06 · Which factor matters, and how well risk can be ranked

**RQ4.** Given route, departure hour and month, how well can delay risk be ranked?

Three models, each with its own honest ceiling: sequential R² to rank factors, a
quantile regression for the tail, and a time-based holdout classifier for risk.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

Team01-FlightDelay-Project v0.2.0 | mode=sample | years=2015-2024 | flights=flights_sample.parquet


In [2]:
from team01 import models, viz

panel = clean.completed(io.load_flights())

ranking = models.incremental_r2(panel)
print(f"model R-squared {ranking.attrs['model_r2']} on {ranking.attrs['rows']:,} sampled flights")
ranking

model R-squared 0.0182 on 46,932 sampled flights


,factor,univariate_r2,cumulative_r2,incremental_r2,share_of_explained
0,dep_hour,0.0116,0.0116,0.0116,0.6409
1,month,0.0019,0.0135,0.0019,0.1050
2,carrier,0.0026,0.0161,0.0026,0.1436
3,origin,0.0020,0.0181,0.0020,0.1105
4,distance,0.0000,0.0182,0.0000,0.0000
5,panel controls (year),NaN,0.0182,0.0000,0.0000


## 6.1 The tail moves differently

A factor can be irrelevant on average and decisive at the ninetieth percentile, so
the same design is fitted at q = 0.9.

In [3]:
quantile = models.quantile_regression(panel, quantile=0.9)
print(f"pseudo R-squared {quantile['pseudo_r2']}")
quantile["table"].head(8)

pseudo R-squared 0.0194


,term,coef
0,dep_hour_22,-31.379679
1,dep_hour_6,-26.676057
2,dep_hour_7,-25.110192
3,dep_hour_5,-22.245877
4,dep_hour_8,-21.282034
5,dep_hour_19,19.744522
6,dep_hour_21,-12.247694
7,dep_hour_15,9.253255


## 6.2 Risk classification on a time holdout

Logistic baseline against histogram gradient boosting. The split is chronological, so
no later month trains an earlier one. Both are reported: a reader needs to see how
much the tree model actually adds, and here it is little — five observable factors
rank risk weakly, and that is the finding.

In [4]:
risk = models.risk_classifier(panel, threshold=60)
print(f"test set starts {risk['split']['actual_test_start']} "
      f"(requested {risk['split']['requested_start']}), "
      f"base rate {risk['base_rate']:.3%}")

pd.DataFrame({name: metrics for name, metrics in risk["models"].items()
              if name != "importance"}).T

test set starts 2024-07-06 (requested 2024-07-01), base rate 8.740%


,roc_auc,pr_auc,brier,base_rate,features
logistic,0.5566,0.1132,0.25931,0.0874,60.0
gradient_boosting,0.5436,0.1015,0.08376,0.0874,NaN


In [5]:
viz.fig_importance(pd.DataFrame(risk["models"]["importance"]))
viz.fig_hour_response(risk["hour_response"], threshold=60)

[viz] 11_importance.png


[viz] 12_hour_response.png


'C:\\Users\\lenovo\\Desktop\\Semester Project\\Team01-FlightDelay-Project\\figures\\12_hour_response.png'